In [ ]:
import os
import json
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from openai import OpenAI

# Importações dos módulos locais
from sqlitesearch import TextSearchIndex
from rag_helper import RAGBase
from evaluation_utils import llm_structured_retry, map_progress, calc_total_price

# 1. Caminhos dos arquivos
DATASET_PATH = Path("data/raw/dataset.json")
BATCH_PATH = Path("data/ground_truth/batch1.json")
OUTPUT_GENERATED_PATH = Path("rag_generated_answers.json")

# 2. Inicialização do Índice Textual Exclusivo
DB_PATH = "mac_faq.db" # Atualizado conforme sua definição
TEXT_FIELDS = ["nome", "pergunta", "resposta"]
KEYWORD_FIELDS = ["agrupamento", "termos", "sinonimos", "sigla"]

text_index = TextSearchIndex(
    text_fields=TEXT_FIELDS,
    keyword_fields=KEYWORD_FIELDS,
    id_field="doc_id",
    db_path=DB_PATH,
)

print("Motor de busca textual (sqlitesearch) carregado com sucesso!")

In [ ]:
# 1. Configuração do Cliente OpenAI apontando para a Groq
api_key = os.environ.get("GROQ_API_KEY", "SUA_CHAVE_AQUI_SE_NAO_ESTIVER_NO_ENV")

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=api_key
)

# 2. Instanciação do Pipeline RAG Simplificado
rag_pipeline = RAGBase(
    text_index=text_index,
    llm_client=client,
    model='llama-3.3-70b-versatile'
)

print("Pipeline RAG inicializado e pronto para geração!")

In [ ]:
# 1. Carregar dataset e criar mapa doc_id -> resposta original
with open(DATASET_PATH, "r", encoding="utf-8") as f:
    dataset = json.load(f)
doc_id_to_resposta = {doc["doc_id"]: doc.get("resposta", "") for doc in dataset}

# 2. Carregar amostra de testes (Batch 1)
with open(BATCH_PATH, "r", encoding="utf-8") as f:
    batch_data = json.load(f)

# 3. Geração das respostas via Pipeline
resultados_gerados = []
rag_pipeline.reset_usage()

for item in tqdm(batch_data, desc="Task 1: Gerando Respostas"):
    pergunta = item["pergunta"]
    doc_id = item["doc_id"]
    resposta_orig = doc_id_to_resposta.get(doc_id, "Resposta original não encontrada.")
    
    # Executa o fluxo RAG (apenas textual)
    resposta_llm = rag_pipeline.rag(pergunta)
    
    resultados_gerados.append({
        "pergunta": pergunta,
        "resposta_llm": resposta_llm,
        "resposta_orig": resposta_orig,
        "doc_id": doc_id
    })

# 4. Salvar resultados
with open(OUTPUT_GENERATED_PATH, "w", encoding="utf-8") as f:
    json.dump(resultados_gerados, f, ensure_ascii=False, indent=4)

print(f"Respostas salvas em: {OUTPUT_GENERATED_PATH}")
custo_geracao = calc_total_price(rag_pipeline.usages)
print(f"Custo total da geração (Task 1): ${custo_geracao:.4f}")

In [ ]:
from concurrent.futures import ThreadPoolExecutor

EVAL_MODEL = "llama-3.3-70b-versatile" 

# Prompts de Instrução para as duas métricas
PROMPT_GROUND_TRUTH = """Você é um avaliador especialista. Compare a "Resposta Gerada" com a "Resposta Original".
Seu objetivo é avaliar o grau de precisão e cobertura da resposta gerada em relação à original.
Regras de pontuação:
- "bom": A resposta gerada cobre todas as informações vitais da original de forma correta.
- "mediano": A resposta gerada está parcialmente correta, mas omite algo importante ou é ambígua.
- "ruim": A resposta gerada está errada, contradiz a original ou sofre de alucinação.

Sua saída deve ser EXCLUSIVAMENTE um objeto JSON válido contendo as chaves:
"score" (com o valor bom, mediano ou ruim) e "reasoning" (uma justificativa curta)."""

PROMPT_ALIGNMENT = """Você é um avaliador especialista. Avalie o quão bem a "Resposta Gerada" atende e responde diretamente à "Pergunta" feita.
Regras de pontuação:
- "bom": Responde à pergunta de forma direta, clara e sem divagações.
- "mediano": Responde à pergunta, mas inclui informações desnecessárias ou é pouco clara.
- "ruim": Não responde à pergunta ou traz informações não relacionadas.

Sua saída deve ser EXCLUSIVAMENTE um objeto JSON válido contendo as chaves:
"score" (com o valor bom, mediano ou ruim) e "reasoning" (uma justificativa curta)."""

# Funções auxiliares para submeter ao ThreadPool
usages_eval = []

def avaliar_ground_truth(item):
    user_prompt = f"Resposta Original: {item['resposta_orig']}\n\nResposta Gerada: {item['resposta_llm']}"
    resultado, usage = llm_structured_retry(client, PROMPT_GROUND_TRUTH, user_prompt, model=EVAL_MODEL)
    usages_eval.append(usage)
    item_copy = item.copy()
    item_copy["avaliacao_ground_truth"] = resultado
    return item_copy

def avaliar_alinhamento(item):
    user_prompt = f"Pergunta: {item['pergunta']}\n\nResposta Gerada: {item['resposta_llm']}"
    resultado, usage = llm_structured_retry(client, PROMPT_ALIGNMENT, user_prompt, model=EVAL_MODEL)
    usages_eval.append(usage)
    item_copy = item.copy()
    item_copy["avaliacao_alinhamento"] = resultado
    return item_copy

In [ ]:
# Carregar dados gerados na Task 1
with open(OUTPUT_GENERATED_PATH, "r", encoding="utf-8") as f:
    generated_data = json.load(f)

WORKERS = 5 # Ajuste conforme os rate limits da sua conta Groq

# 1. Avaliação 1: Ground Truth
print("Iniciando Avaliação: Ground Truth (Comparação com a Original)")
with ThreadPoolExecutor(max_workers=WORKERS) as pool:
    resultados_gt = map_progress(pool, generated_data, avaliar_ground_truth)

with open("evaluation_ground_truth.json", "w", encoding="utf-8") as f:
    json.dump(resultados_gt, f, ensure_ascii=False, indent=4)

# 2. Avaliação 2: Alinhamento
print("\nIniciando Avaliação: Alinhamento (Pergunta vs Resposta)")
with ThreadPoolExecutor(max_workers=WORKERS) as pool:
    resultados_qa = map_progress(pool, generated_data, avaliar_alinhamento)

with open("evaluation_alinhamento.json", "w", encoding="utf-8") as f:
    json.dump(resultados_qa, f, ensure_ascii=False, indent=4)

print("\nAvaliações concluídas e arquivos JSON salvos!")
custo_eval = calc_total_price(usages_eval)
print(f"Custo total da avaliação (Task 2): ${custo_eval:.4f}")

In [ ]:
# Unificando os dados para visualização em DataFrame
df_gt = pd.DataFrame(resultados_gt)
df_qa = pd.DataFrame(resultados_qa)

# Extraindo os scores dos dicionários
df_gt['score_gt'] = df_gt['avaliacao_ground_truth'].apply(lambda x: x.get('score'))
df_qa['score_qa'] = df_qa['avaliacao_alinhamento'].apply(lambda x: x.get('score'))

print("--- Distribuição de Scores (Ground Truth) ---")
print(df_gt['score_gt'].value_counts())

print("\n--- Distribuição de Scores (Alinhamento) ---")
print(df_qa['score_qa'].value_counts())

# Exibindo os 5 primeiros resultados consolidados
df_final = df_gt[['pergunta', 'doc_id', 'score_gt']].copy()
df_final['score_qa'] = df_qa['score_qa']
df_final.head()